# Visualizing time series data

```
Authors: Thomas Moreau
         Mathurin Massias
         Alexandre Gramfort
```

Licensed under [CC BY 4.0 Creative Commons](http://creativecommons.org/licenses/by/4.0/)

In [ ]:
import pandas as pd

Some imports

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('data/AirPassengers.csv')
df.rename(columns={'Month': 'Date'}, inplace=True)
df["Date"] = pd.to_datetime(df["Date"])
df

## Line plots

In [ ]:
df.loc[:"1950"].plot(x='Date', y='Passengers')
# or df.set_index('Date').plot()

In [ ]:
sns.lineplot(
    data=df, x='Date', y='Passengers'
).set_title('Airline passengers over time');

## Seasonal plot

In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.strftime('%b')

sns.lineplot(
    data=df, x='Month', y='Passengers',
    hue='Year', legend='full'
)

# add title
plt.title('Seasonal plot')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2);

## Month plot (and quarter plot)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf, month_plot, quarter_plot

df = df.set_index("Date")
df.index = pd.DatetimeIndex(df.index, freq='MS')
month_plot(df["Passengers"], ylabel='Passengers');

In [ ]:
quarter_plot(df["Passengers"].resample('Q').sum(), ylabel='Passengers');

## Polar seasonal plot

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
df["Month_polar"] = df.index.month / 12 * 2 * np.pi
sns.lineplot(data=df, x="Month_polar", y='Passengers', hue="Year", ax=ax)
ax.set_xticks(np.arange(0, 2 * np.pi, np.pi / 6))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.set_title('Passengers over time')
ax.set_ylabel("")
plt.legend(bbox_to_anchor=(1.05, 1), loc=2);

You can also do this with Plotly

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import plotly.express as px

fig = px.line_polar(df, r='Passengers', theta='Month',
                    color='Year', line_close=True,
                    title='Polar seasonal plot',
                    width=600, height=500)
fig.show()

## Lag plots

Lag plots are most commonly used to look for patterns in time series data.

In [ ]:
plt.plot(df["Passengers"], df["Passengers"].shift(1), 'o')

Or using Pandas plotting tools:

In [ ]:
lags = range(1, 13)
fig, axes = plt.subplots(3, 4, figsize=(12, 10))
for lag, ax in zip(lags, axes.ravel()):
    pd.plotting.lag_plot(df["Passengers"], lag=lag, ax=ax)
    ax.set_title(f'Lag {lag}')
plt.tight_layout()

## Autocorrelation plots

Autocorrelation is computing the correlation of a signal with a delayed copy of itself as a function of delay (lag).

In [ ]:
plot_acf(df['Passengers']);
plt.grid("on")

In the ACF plot, we can see that there are **significant autocorrelations** (above the 95% confidence interval, corresponding to the default 5% significance level).

The slow decrease in the ACF as the lags increase is due to the trend, while the "scalloped" shape is due the seasonality.

Let's now look at the **partial autocorrelation function (PACF)**.

In [ ]:
plot_pacf(df['Passengers']);
plt.grid("on")

PACF looks at the correlation between the residuals (the remainder after removing the effects explained by the previous lags) and the following lag value. This way, we effectively remove the already found variations before we find the next correlation. In practice, a high partial correlation indicates that there is **some information in the residual that can be modeled by the next lag**. So we **might consider keeping that lag as a feature in our model**.

Let's see how these plots would look like for white noise:

In [ ]:
rng = np.random.default_rng(42)
white_noise = rng.normal(size=300)

fig, axes = plt.subplots(2, 1, figsize=(8, 6))
plot_acf(white_noise, ax=axes[0])
plot_pacf(white_noise, ax=axes[1])
plt.tight_layout()

<div class="alert alert-warning">

**Careful:** Non-negative partial autocorrelations are not to be interpreted as
a sign of the frequency of the signal. They are a measure of which lags are
useful to predict the signal.

</div>

In [ ]:
sin = pd.DataFrame({'date': pd.date_range(start='1/1/1949', periods=len(df), freq='MS')})
sin['value'] = np.sin(2 * np.pi / 12 * sin.date.dt.month)
sin = sin.set_index('date')

fig, axes = plt.subplots(3, 1, figsize=(6, 6))
sin.plot(ax=axes[0])
plot_acf(sin, ax=axes[1])
plot_pacf(sin, ax=axes[2])
plt.tight_layout()

<div class="alert alert-success">

**QUESTION:**

- Propose and implement a method to remove the trend in the data and plot the autocorrelation and partial autocorrelation functions of the detrended data.

</div>

Solution is in: `solutions/visu-1_ts-detrending.py`

In [ ]:
from sklearn.linear_model import LinearRegression

trend = ...

residual = df["Passengers"] - trend

fig, axes = plt.subplots(4, 1, figsize=(8, 6))
axes[0].plot(df["Passengers"])
axes[0].plot(trend, "r")
axes[1].plot(residual)
axes[1].set_title("Residuals")
plot_acf(residual, ax=axes[2])
plot_pacf(residual, ax=axes[3])
plt.tight_layout()


# Evaluating a model with autocorrelated time-series

Let's take an example of some financial quotes

> **Think before running:** what features does the model have access to when making a prediction? Is it realistic?

Here we look at four oil & gas companies — ExxonMobil (XOM), Chevron (CVX), ConocoPhillips (COP), Valero (VLO). Because they operate in the same sector, their stock prices tend to move together. The goal is to predict Chevron's price from the others.

In [ ]:
from utils import fetch_quote_data
quotes = fetch_quote_data(['XOM', 'CVX', 'COP', 'VLO'])

In [ ]:
_, ax = plt.subplots(figsize=(10, 6))
_ = quotes.plot(ax=ax)

Let's assume we want to predict the stock price of "Chevron" from the stock price of the other companies at each time point.

In [ ]:
quotes

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

X, y = quotes.drop(columns=["Chevron"]), quotes["Chevron"]
regressor = GradientBoostingRegressor()

In [ ]:
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score

cv = ShuffleSplit(n_splits=30)
scores = cross_val_score(regressor, X, y, cv=cv)
print(f'Mean R2: {scores.mean():.2f}')

<div class="alert alert-success">

**QUESTION:**

- It seems that we have the perfect regressor. Is this normal? You will use the function `statsmodels.tsa.stattools.acf` look at the correlation between sucessive samples.
- Did we break any assumption of the cross-validation procedure?

</div>

Solution is in: `solutions/visu-2_time-series-split.py`

Let's check the different type of cross-validation that are available in scikit-learn:

https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html#sphx-glr-auto-examples-model-selection-plot-cv-indices-py